In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from pymargins import GComputation

rng = np.random.default_rng(42)
n = 1000
df = pd.DataFrame({
    "age": rng.integers(18, 80, size=n),
    "treat": rng.binomial(1, 0.4, size=n),
    "x1": rng.normal(size=n),
})
logit_p = 1 / (1 + np.exp(-(-1.0 + 0.04 * df["age"] + 0.5 * df["treat"] + 0.3 * df["x1"])))
df["y"] = rng.binomial(1, logit_p)

model = smf.logit("y ~ treat*x1 + age", data=df).fit(disp=0)
est = GComputation(model, at="overall", scale="response", method="delta")

est.predict()                                    # average adjusted prediction
est.predict(atexog={"treat": [0, 1]})            # standardized rates
est.contrasts(
    scenarios=[{"atexog": {"treat": 0}}, {"atexog": {"treat": 1}}],
    contrasts=[-1, 1],
)                                                  # marginal risk difference
est.dydx("age")                                  # AME

GraphResult(estimate=0.006138, std_error=0.000626, conf_int=[0.004912, 0.007365], labels=['age'], method='delta', scale='response', level=0.95, ci='wald', n_obs=1000, kappa=0.066132, gradient=[-2.685547e-03, -1.098633e-03, -4.272461e-04, -6.103516e-05,  1.562500e-02], cov_params=array(shape=(5, 5), dtype=float32), psi_h=array(shape=(1000,), dtype=float32))

In [2]:
# Requires: pip install lifelines
from lifelines import CoxPHFitter

rng = np.random.default_rng(7)
n = 800
df_surv = pd.DataFrame({
    "time": rng.exponential(50, size=n),
    "event": rng.binomial(1, 0.8, size=n),
    "treat": rng.binomial(1, 0.4, size=n),
    "age": rng.normal(50, 10, size=n),
})

cph = CoxPHFitter().fit(df_surv, "time", "event")

from pymargins.adapters import LifelinesCoxPHAdapter

est = GComputation(
    cph,
    adapter=LifelinesCoxPHAdapter(cph, training_data=df_surv),
    at="overall",
    scale="response",
    method="simulation",
    n_sim=2000,
    seed=42,
)

est.dydx("age")

GraphResult(estimate=0.003386, std_error=0.004133, conf_int=[-0.004817, 0.011425], labels=['age'], method='simulation', scale='response', level=0.95, ci='wald', n_obs=800, kappa=0.004086, draws=array(shape=(2000,), dtype=float32), draws_inf=array(shape=(2000,), dtype=float32))

In [3]:
import logging

from pymargins import steps, PysmatchClient
from pysmatch.Matcher import Matcher

# pysmatch logs ~15 INFO/WARNING lines (to the root logger) on every match,
# and the bootstrap below re-matches on each of the B replicates. Raise the
# root log level so that per-replicate spam stays out of the rendered output
# and the cached notebook.
logging.getLogger().setLevel(logging.ERROR)

test = df[df["treat"] == 1].copy()
control = df[df["treat"] == 0].copy()
matcher = Matcher(test, control, yvar="treat", exclude=["y"])
matcher.fit_scores(balance=True, model_type="linear")
matcher.predict_scores()
matcher.match(method="min", nmatches=1, threshold=0.001)

matched = matcher.matched_data
model = smf.logit("y ~ treat + x1 + age", data=matched).fit(disp=0)

est = GComputation(
    steps.match(steps.input(matched), PysmatchClient(matcher, treatment_col="treat")),
    outcome=model,
    at="overall",
    scale="response",
    method="bootstrap",
    B=999,
    seed=123,
)
est.contrasts(
    scenarios=[{"atexog": {"treat": 0}}, {"atexog": {"treat": 1}}],
    contrasts=[-1, 1],
)

GraphResult(estimate=0.065824, std_error=0.033736, conf_int=[-0.003661, 0.128395], labels=['contrast'], method='bootstrap', scale='response', level=0.95, ci='percentile', n_obs=748, kappa=0.047776, draws=array(shape=(999,), dtype=float32), draws_inf=array(shape=(999,), dtype=float32))

In [4]:
print(est.plan.hash)
print(est.plan.describe())

186890a@1
Plan 186890a@1
  method: bootstrap (declared: bootstrap)
  resolution reason: user-specified
  scale: response
  at: overall
  ci: percentile
  level: 0.95
  B: 999
  n_sim: 4000
  seed: 123
  data fingerprint: 7b0a16a760dc51da...
